In [ ]:
!pip install googletrans==4.0.0rc1
!pip install -U sentence-transformers

In [ ]:
import torch
from googletrans import Translator
#from transformers import RobertaModel, RobertaTokenizer
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import time

In [ ]:
translator = Translator()

In [ ]:
#model_name = 'roberta-base'
#tokenizer = RobertaTokenizer.from_pretrained(model_name)
#model = RobertaModel.from_pretrained(model_name)
model = SentenceTransformer("multi-qa-mpnet-base-cos-v1")

In [ ]:
audio_dir = "/kaggle/input/recognized-audio-second-half/archive"
output_dir = "/kaggle/working/31 audio features"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.txt')]
for audio_file in audio_files:
    if audio_file.startswith('L25'):
        audio_path = os.path.join(audio_dir, audio_file)
        audio_id = os.path.splitext(audio_file)[0]
        print(f"Processing video: {audio_id}")
        try:
            with open(audio_path, 'r', encoding='utf-8') as file:
                content = file.read()
            translated = translator.translate(content, dest='en')
            content = translated.text
            '''
            inputs = tokenizer(content, return_tensors='pt', truncation=True, padding=True)
            with torch.no_grad():
                outputs = model(**inputs)
            feature_vector = np.array(outputs.last_hidden_state[:, 0, :]).squeeze()
            '''
            feature_vector = np.array(model.encode(content)).squeeze()
            feature_vector = feature_vector / np.linalg.norm(feature_vector)  
        
            '''
            if (audio_id == 'L01_V029_150.0_211.0'):
                print(content)
                print(features)
            '''
            output_path = os.path.join(output_dir, f"{audio_id}.npy")
            np.save(output_path, feature_vector)
            print(f"Saved features for {audio_id} to {output_path}")
            time.sleep(0.2)
        except Exception as e:
            print(f"Error processing video {audio_id}: {e}")